[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-processing/N2N-SPRVS/blob/main/N2N-SPRVS.ipynb)

# N2N-SPRVS (Noise2Noise Structure Preserving Random Voxel) tomogram denoising

In [ ]:
REO_folder="~/REO/ROIs/empiar11415_ROI_718,1217_947,1446_1,56/"

In [ ]:
std_dev = 3.5

In [ ]:
gpu_id=0

In [ ]:
N = 1  # >=0 Number of generated tomograms used for training the denoiser

In [ ]:
!nvidia-smi

## Packages

In [ ]:
# %pip install cupy-cuda12x # Check the CUDA installed with nvcc --version (that should match with the version provided by nvidia-smi)
# %pip install numba
# %pip install tqdm ipywidgets
# %pip install scipy
# %pip install "optical_flow_3D @ git+https://github.com/vicente-gonzalez-ruiz/optical_flow_3D"
#import optical_flow_3D
import cv2

In [ ]:
#%pip install mrcfile
import mrcfile

In [ ]:
# %pip install numpy
import numpy as np

In [ ]:
# %pip install tqdm ipywidgets
from tqdm.notebook import tqdm

In [ ]:
# %pip install matplotlib
import matplotlib.pyplot as plt

In [ ]:
import json

In [ ]:
# %pip install gdown
import gdown

In [ ]:
from pathlib import Path

In [ ]:
import os

## Download a noisy tomogram

In [ ]:
url="https://drive.google.com/file/d/1UtFnrgTj0JE0wF3GfPNF8BBWat83j0U_"
#gdown.download(url, output="000.mrc", quiet=False, use_cookies=False)

In [ ]:
X = mrcfile.open("000.mrc").data

In [ ]:
X.shape

## Generate another version of the tomogram

In [ ]:
farneback_params = dict(
    pyr_scale=0.5,
    levels=3,
    winsize=15,
    iterations=3,
    poly_n=5,
    poly_sigma=1.2,
    flags=0
)

In [ ]:
# https://stackoverflow.com/questions/62436299/how-to-lightly-shuffle-a-list-in-python
orderliness = 0.75

def tuplify(x, y):
  return (orderliness * y + np.random.normal(0, 1), x)

def shake(x, y, std_dev=1.0):
  displacements = np.random.normal(0, std_dev, len(x))
  #print(f"{np.min(displacements):.2f} {np.average(np.abs(displacements)):.2f} {np.max(displacements):.2f}", end=' ')
  return np.stack((y + displacements, x), axis=1)

def randomize(slice, mean=0.0, std_dev=1.0):
  #print(slice.shape)
  #print(std_dev)
  randomized_slice = np.empty_like(slice)

  # Randomization in Y
  values = np.arange(slice.shape[0]).astype(np.int32)
  for x in range(slice.shape[1]):
    #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
    pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
    pairs = pairs[pairs[:, 0].argsort()]
    randomized_slice[values, x] = slice[pairs[:, 1], x]

  # Randomization in X
  values = np.arange(slice.shape[1]).astype(np.int32)
  for y in range(slice.shape[0]):
    #pairs = np.array(list(map(tuplify, values, range(len(values)))), dtype=np.int32)
     pairs = shake(values, np.arange(len(values)), std_dev).astype(np.int32)
     pairs = pairs[pairs[:, 0].argsort()]
     randomized_slice[y, values] = randomized_slice[y, pairs[:, 1]]

  return randomized_slice

In [ ]:
def normalize(x):
    x_min, x_max = x.min(), x.max()
    return (255.0 * (x - x_min) / (x_max - x_min)).astype(np.uint8)

In [ ]:
def generate_similar(X, std_dev=2.0):
    similar_X = np.zeros_like(X, dtype=np.float32)
    for z in tqdm(range(X.shape[0]), desc=f"std_dev={std_dev}"):
        original_slice = X[z]
        shaked_slice = randomize(slice=original_slice, std_dev=std_dev)

        # Calculate the dense optical flow from slice_z_plus_1 to slice_z
        flow = cv2.calcOpticalFlowFarneback(normalize(original_slice), normalize(shaked_slice), None, **farneback_params)

        # Create a remapping grid from the flow field
        height, width = flow.shape[:2]
        x_coords, y_coords = np.meshgrid(np.arange(width), np.arange(height))

        # The new map tells where each pixel in the output image should come from in the input image
        map_x = (x_coords + flow[..., 0]).astype(np.float32)
        map_y = (y_coords + flow[..., 1]).astype(np.float32)

        # Warp the *original float32 slice* using the map for maximum precision
        projected_slice = cv2.remap(
            src=original_slice,
            map1=map_x,
            map2=map_y,
            #interpolation=cv2.INTER_LINEAR,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_REPLICATE # Handle edge pixels
        )

        # Store the result
        similar_X[z, ...] = projected_slice
    return similar_X

In [ ]:
for i in range(N):
    similar_X = generate_similar(X, std_dev=std_dev)
    output_filename = f"{i+1:03d}.mrc"
    with mrcfile.new(output_filename, overwrite=True) as mrc:
        mrc.set_data(similar_X)
        mrc.data

In [ ]:
similar_X.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 3, figsize=(20, 20))

im0 = axes[0].imshow(X[slice_idx, ...], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice z={slice_idx}')
axes[0].grid(False)

#im1 = axes[1].imshow(shuffled_X[slice_idx, ...], cmap='gray', origin='lower')
#axes[1].set_title(f'Shuffled Slice z={slice_idx}')
#axes[1].grid(False)

im1 = axes[1].imshow(similar_X[slice_idx, ...], cmap='gray', origin='lower')
axes[1].set_title(f'Projected Slice z={slice_idx}')
axes[1].grid(False)

#im3 = axes[3].imshow((X[slice_idx, ...] - similar_X[slice_idx, ...] + 128).astype(np.int16) , cmap='gray', origin='lower')
im2 = axes[2].imshow((X[slice_idx, ...] - similar_X[slice_idx, ...] + 128).astype(np.int16), cmap='gray', origin='lower')
#im3 = axes[3].imshow(((X[slice_idx, ...] != similar_X[slice_idx, ...]) * 255), cmap='gray', origin='lower')
axes[2].set_title(f'original[z] - projected[z]')
axes[2].grid(False)

plt.tight_layout()
plt.show()

# Denoising

In [ ]:
# %pip install tensorflow

In [ ]:
# %pip install cryoCARE --no-deps

In [ ]:
# %pip install csbdeep

In [ ]:
_ = {
    "even": ["noisy_vol.mrc"],
    "odd": ["shuffled_X.mrc"],
    "mask": [""],
    "patch_shape": [8, 8, 8], # <- Be careful here: in this example the tomogram is very small and the patch shape must be also small
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_SPRVS",
    "overwrite": "True"
}

_ = {
    "even": ["000.mrc", "002.mrc", "004.mrc", "006.mrc"],
    "odd": ["001.mrc", "003.mrc", "005.mrc", "007.mrc"],
    "mask": [""],
    "patch_shape": [16, 16, 16],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data_SPRVS",
    "overwrite": "True"
}

def generate_cryocare_config(N):
    #even_tomograms = []
    generated_tomograms = []
    for i in range(N):
        filename = f"{i+1:03d}.mrc"
        # Sort into even or odd lists
        #if i % 2 == 0:
        #    even_tomograms.append(filename)
        #else:
        generated_tomograms.append(filename)

    # Construct the final configuration dictionary
    config = {
        "even": even_tomograms,
        "odd": odd_tomograms,
        "mask": [""],
        "patch_shape": [16, 16, 16],
        "num_slices": 800,
        "split": 0.9,
        "tilt_axis": "Y",
        "n_normalization_samples": 200,
        "path": "./data_SPRVS",
        "overwrite": "True"
    }

    # Construct the final configuration dictionary
    config = {
        "even": ["000.mrc"]*(N),
        "odd": generated_tomograms,
        "mask": [""],
        "patch_shape": [16, 16, 16],
        "num_slices": 800,
        "split": 0.9,
        "tilt_axis": "Y",
        "n_normalization_samples": 200,
        "path": "./data_SPRVS",
        "overwrite": "True"
    }

    return config

def generate_cryocare_config(N):
    even_tomograms = ["000.mrc"]
    odd_tomograms = []
    for i in range(1,N+1):
        filename = f"{i:03d}.mrc"
        # Sort into even or odd lists
        if i % 2 == 0:
            even_tomograms.append(filename)
        else:
            odd_tomograms.append(filename)

    # Construct the final configuration dictionary
    config = {
        "even": even_tomograms,
        "odd": odd_tomograms,
        "mask": [""],
        "patch_shape": [8, 8, 8],
        "num_slices": 800,
        "split": 0.9,
        "tilt_axis": "Y",
        "n_normalization_samples": 200,
        "path": "./data_SPRVS",
        "overwrite": "True"
    }

    return config

_ = generate_cryocare_config(N)

with open("train_data_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat train_data_config__SPRVS.json

In [ ]:
%%bash
/home/jupyter-vruiz/envs/OF3D_CUDA/bin/cryoCARE_extract_train_data.py --conf train_data_config__SPRVS.json

%%writefile train_config__SPRVS.json
{
  "train_data": "./data_SPRVS",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_SPRVS",
  "path": "./",
  "gpu_id": [2]
}

In [ ]:
_ = {
  "train_data": "./data_SPRVS",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model_SPRVS",
  "path": "./",
  "gpu_id": [gpu_id]
}
with open("train_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
/home/jupyter-vruiz/envs/OF3D_CUDA/bin/cryoCARE_train.py --conf train_config__SPRVS.json

In [ ]:
_ = {
    "path": "./model_SPRVS.tar.gz",
    "even": ["000.mrc"],
    "odd": ["001.mrc"],
    "n_tiles": [1,1,1],
    "output": "denoised_vol_SPRVS",
    "overwrite": "True",
    "gpu_id": [1]
}


_ = {
    "path": "./model_SPRVS.tar.gz",
    "even": ["000.mrc"],
    "odd": ["001.mrc"],
    "n_tiles": [1,1,1],
    "output": "denoised_vol_SPRVS",
    "overwrite": "True",
    "gpu_id": [1]
}

def generate_cryocare_config(N):
    even_tomograms = ["000.mrc"]
    odd_tomograms = []
    for i in range(1,N+1):
        filename = f"{i:03d}.mrc"
        # Sort into even or odd lists
        if i % 2 == 0:
            even_tomograms.append(filename)
        else:
            odd_tomograms.append(filename)

    # Construct the final configuration dictionary
    config = {
        "path": "./model_SPRVS.tar.gz",
        "even": even_tomograms,
        "odd": even_tomograms,
        "n_tiles": [2,2,4],
        "output": "denoised_vol_SPRVS",
        "overwrite": "True",
        "gpu_id": [gpu_id]
    }

    return config

_ = generate_cryocare_config(N)

with open("predict_config__SPRVS.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
!cat predict_config__SPRVS.json

In [ ]:
%%bash
/home/jupyter-vruiz/envs/OF3D_CUDA/bin/cryoCARE_predict.py --conf predict_config__SPRVS.json || true

In [ ]:
!ls -l denoised_vol_SPRVS/*

In [ ]:
!ls -l $REO_folder/denoised_vol_REO/000.mrc

In [ ]:
REO = mrcfile.open(os.path.expanduser(f"{REO_folder}denoised_vol_REO/000.mrc")).data

In [ ]:
import numpy as np
import mrcfile

file_path = os.path.expanduser(f"{REO_folder}denoised_vol_REO/000.mrc")

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
REO = data.reshape((nz, ny, nx))

sum = np.copy(mrcfile.open(f"denoised_vol_SPRVS/000.mrc").data)
print(sum.dtype)
N = 0
for i in range((N>>1)):
  print(i)
  sum += mrcfile.open(f"denoised_vol_SPRVS/{2*i:03d}.mrc").data
if N>1:
    average = sum/(N>>1)
else:
    average = sum

with mrcfile.new("denoised_vol_SPRVS/average.mrc", overwrite=True) as mrc:
  mrc.set_data(average)
  mrc.data

In [ ]:
#Y = mrcfile.read("denoised_vol_SPRVS/noisy_vol.mrc")
#Y = mrcfile.read("denoised_vol_SPRVS/average.mrc")
Y = mrcfile.read("denoised_vol_SPRVS/000.mrc")

In [ ]:
import numpy as np
import mrcfile

file_path = "denoised_vol_SPRVS/000.mrc"

with mrcfile.open(file_path, permissive=True) as m:
    nx = m.header.nx
    ny = m.header.ny
    nz = m.header.nz
    nsymbt = m.header.nsymbt

# compute data offset
offset = 1024 + nsymbt

# read raw bytes manually
data = np.fromfile(
    file_path,
    dtype=np.float32,   # <-- force desired dtype
    offset=offset
)

# reshape manually
Y = data.reshape((nz, ny, nx))

In [ ]:
X.shape

In [ ]:
Y.shape

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx, :, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(X[slice_idx, :, :], cmap='gray', origin='lower')
axes[1].set_title(f'Original Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(X[slice_idx, :, :], cmap='gray', origin='lower')
axes.set_title(f'Original Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('original.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx, :, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(REO[slice_idx, :, :], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-REO Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(REO[slice_idx, :, :], cmap='gray', origin='lower')
axes.set_title(f'N2N-REO Slice Z={slice_idx}')
axes.grid(False)

plt.tight_layout()

plt.savefig('REO.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[slice_idx, :, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[slice_idx, :, :], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-SPRVS Slice Z={slice_idx}, std_dev={std_dev}')
axes[1].grid(False)

plt.tight_layout()

plt.show()

In [ ]:
slice_idx = X.shape[0] // 2

fig, axes = plt.subplots(1, 1, figsize=(15, 15))

# Plot a denoised slice
im2 = axes.imshow(Y[slice_idx, :, :], cmap='gray', origin='lower')
axes.set_title(f'N2N-SPRVS Slice Z={slice_idx}, std_dev={std_dev}')
axes.grid(False)

plt.tight_layout()

plt.savefig('SPRVS.png', dpi=75, bbox_inches='tight')

plt.show()

In [ ]:
slice_idx = X.shape[1] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, slice_idx, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Y={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(REO[:, slice_idx, :], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-REO Slice Y={slice_idx}, std_dev={std_dev}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[1] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, slice_idx, :], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Y={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[:, slice_idx, :], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-SPRVS Slice Y={slice_idx}, std_dev={std_dev}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, :, slice_idx], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice X={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(REO[:, :, slice_idx], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-REO Slice X={slice_idx}, std_dev={std_dev}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
slice_idx = X.shape[2] // 2

fig, axes = plt.subplots(1, 2, figsize=(15, 15))

# Plot a original slice
im1 = axes[0].imshow(X[:, :, slice_idx], cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice X={slice_idx}')
axes[0].grid(False)

# Plot a denoised slice
im2 = axes[1].imshow(Y[:, :, slice_idx], cmap='gray', origin='lower')
axes[1].set_title(f'N2N-SPRVS Slice X={slice_idx}, std_dev={std_dev}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
# %pip install --force-reinstall --no-cache-dir "self_fourier_shell_correlation @ git+https://github.com/vicente-gonzalez-ruiz/self_fourier_shell_correlation"
from self_fourier_shell_correlation import fsc_utils as fsc

In [ ]:
# %pip show self_fourier_shell_correlation

In [ ]:
# %pip install  --force-reinstall --no-cache-dir "shuffling @ git+https://github.com/vicente-gonzalez-ruiz/shuffling"

In [ ]:
!ls -l

In [ ]:
list_fsc_values__X = []
list_fsc_values__REO = []
list_fsc_values__SPRVS = []
for i in range(X.shape[0]-2):
    print(i, '/', X.shape[0])
    spatial_freqs, fsc_values__X = fsc.get_SFRC_curve__subsampled_chessboard(X[i])
    spatial_freqs, fsc_values__REO = fsc.get_SFRC_curve__subsampled_chessboard(REO[i])
    spatial_freqs, fsc_values__SPRVS = fsc.get_SFRC_curve__subsampled_chessboard(Y[i])
    list_fsc_values__X.append(fsc_values__X)
    list_fsc_values__REO.append(fsc_values__REO)
    list_fsc_values__SPRVS.append(fsc_values__SPRVS)

In [ ]:
avg_fsc_values__X = np.mean(list_fsc_values__X, axis=0)
avg_fsc_values__REO = np.mean(list_fsc_values__REO, axis=0)
avg_fsc_values__SPRVS = np.mean(list_fsc_values__SPRVS, axis=0)

In [ ]:
plt.title(f"{Path.cwd().parts[-2:]}, std_dev={std_dev}")
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__X, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__REO, label="N2N-REO", color="red")
plt.plot(spatial_freqs, avg_fsc_values__SPRVS, label="N2N-SPRVS", color="green")
plt.legend(loc='lower left')

list_fsc_values__X = []
list_fsc_values__REO = []
list_fsc_values__SPRVS = []
for i in range(X.shape[1]):
    print(i, '/', X.shape[2])
    spatial_freqs, fsc_values__X = fsc.get_SFRC_curve__subsampled_chessboard(X[:,i,:])
    spatial_freqs, fsc_values__REO = fsc.get_SFRC_curve__subsampled_chessboard(REO[:,i,:])
    spatial_freqs, fsc_values__SPRVS = fsc.get_SFRC_curve__subsampled_chessboard(Y[:,i,:])
    list_fsc_values__X.append(fsc_values__X)
    list_fsc_values__REO.append(fsc_values__REO)
    list_fsc_values__SPRVS.append(fsc_values__SPRVS)

avg_fsc_values__X = np.mean(list_fsc_values__X, axis=0)
avg_fsc_values__REO = np.mean(list_fsc_values__REO, axis=0)
avg_fsc_values__SPRVS = np.mean(list_fsc_values__SPRVS, axis=0)

plt.title(Path.cwd().parts[-2:])
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__X, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__REO, label="N2N-REO", color="red")
plt.plot(spatial_freqs, avg_fsc_values__SPRVS, label="N2N-SPRVS", color="green")
plt.legend(loc='lower left')

list_fsc_values__X = []
list_fsc_values__REO = []
list_fsc_values__SPRVS = []
for i in range(X.shape[2]):
    print(i, '/', X.shape[2])
    spatial_freqs, fsc_values__X = fsc.get_SFRC_curve__subsampled_chessboard(X[:,:,i])
    spatial_freqs, fsc_values__REO = fsc.get_SFRC_curve__subsampled_chessboard(REO[:,:,i])
    spatial_freqs, fsc_values__SPRVS = fsc.get_SFRC_curve__subsampled_chessboard(Y[:,:,i])
    list_fsc_values__X.append(fsc_values__X)
    list_fsc_values__REO.append(fsc_values__REO)
    list_fsc_values__SPRVS.append(fsc_values__SPRVS)

avg_fsc_values__X = np.mean(list_fsc_values__X, axis=0)
avg_fsc_values__REO = np.mean(list_fsc_values__REO, axis=0)
avg_fsc_values__SPRVS = np.mean(list_fsc_values__SPRVS, axis=0)

plt.title(Path.cwd().parts[-2:])
plt.xlabel("Normalized Spatial Frequency (cycles/pixel)")
plt.ylabel("Average Self Fourier Ring Correlation")
plt.plot(spatial_freqs, avg_fsc_values__X, label="Noisy", color="blue")
plt.plot(spatial_freqs, avg_fsc_values__REO, label="N2N-REO", color="red")
plt.plot(spatial_freqs, avg_fsc_values__SPRVS, label="N2N-SPRVS", color="green")
plt.legend(loc='lower left')

In [ ]:
import scipy.stats

In [ ]:
def PCC(original, denoised):
    return scipy.stats.pearsonr(original.flatten(), denoised.flatten())[0]

In [ ]:
print(f"PCC(std_dev={std_dev})={PCC(X, Y):.3f}")

In [ ]:
#%pip install scikit-image
import skimage.metrics

In [ ]:
def PSNR(original, denoised):
    return skimage.metrics.peak_signal_noise_ratio(original, denoised, data_range=original.max()-original.min())

In [ ]:
print(f"PSNR(std_dev={std_dev})={PSNR(X, Y):.3f}")

In [ ]:
def SSIM(original, denoised):
    return skimage.metrics.structural_similarity(original, denoised, data_range=original.max() - original.min())

In [ ]:
print(f"SSIM(std_dev={std_dev})={SSIM(X, Y):.3f}")